# Data Processing

## Download the Data

In [60]:
# Kaggle Notebook 数据读取模板
from pathlib import Path
import pandas as pd

def load_housing_data():
    data_path = Path("/kaggle/input/housing-data/housing_data") 
    dfs = {
        "train_price": pd.read_csv(data_path / "train_price.csv"),
        "train_rent": pd.read_csv(data_path / "train_rent.csv"),
        "test_price": pd.read_csv(data_path / "test_price.csv"),
        "test_rent": pd.read_csv(data_path / "test_rent.csv")
    }

    return dfs

housing = load_housing_data()

/tmp/ipykernel_37/951388431.py:8: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  "train_price": pd.read_csv(data_path / "train_price.csv"),
/tmp/ipykernel_37/951388431.py:9: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  "train_rent": pd.read_csv(data_path / "train_rent.csv"),
/tmp/ipykernel_37/951388431.py:10: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  "test_price": pd.read_csv(data_path / "test_price.csv"),


## Data Cleaning

### train_price

In [61]:
train_price = housing['train_price'].copy()
train_price = train_price[train_price['别墅类型'].isna()]
train_price.drop(columns=['梯户比例','套内面积','环线位置','房屋朝向','别墅类型','交易时间','交易权属','上次交易','房屋用途','产权所属','抵押信息','房屋优势','核心卖点','户型介绍','周边配套','交通出行','年份','区县','板块_comm','物业类别','建筑年代','开发商','房屋总数','楼栋总数','物业公司','建筑结构_comm','装修情况','物业办公电话','产权描述','供水','供暖','供电','coord_x','coord_y','客户反馈'], inplace=True)

In [62]:
train_price.drop_duplicates(inplace=True)
train_price.dropna(subset=['配备电梯'],inplace=True)
train_price['环线'] = train_price['环线'].fillna('未知')

In [63]:
train_price.rename(columns={
    '建筑面积': '建筑面积（㎡）',
    '绿 化 率': '绿化率（%）',
    '容 积 率': '容积率（倍）',
    '物 业 费': '物业费（元/月/㎡）',
    '燃气费': '燃气费（元/m³）',
    '供热费': '供热费（元/㎡）',
    '停车位': '停车位（个）',
    '停车费用': '停车费用（元）',
}, inplace = True)
train_price['绿化率（%）'] = train_price['绿化率（%）'].astype(str).str.replace('%', '', regex=False)
train_price['绿化率（%）'] = pd.to_numeric(train_price['绿化率（%）'], errors='coerce')
train_price['建筑面积（㎡）'] = train_price['建筑面积（㎡）'].astype(str).str.replace('㎡', '', regex=False)
train_price['建筑面积（㎡）'] = pd.to_numeric(train_price['建筑面积（㎡）'], errors='coerce')
train_price['燃气费（元/m³）'] = train_price['燃气费（元/m³）'].astype(str).str.replace('元/m³', '', regex=False)
train_price['供热费（元/㎡）'] = train_price['供热费（元/㎡）'].astype(str).str.replace('元/㎡', '', regex=False)
train_price['物业费（元/月/㎡）'] = train_price['物业费（元/月/㎡）'].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
# [^\d\.-]匹配所有不是数字、不是小数点、不是减号的字符

In [64]:
# 将区间数据用平均值代替
import pandas as pd
import numpy as np

cols = ['物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）']
for col in cols:
    train_price[col] = train_price[col].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
    train_price[col] = train_price[col].replace('', np.nan)
  
    def parse_range(x):
        if pd.isna(x):
            return np.nan
        parts = x.split('-')
        return (float(parts[0]) + float(parts[1])) / 2 if len(parts) > 1 else float(parts[0])
    
    train_price[col] = train_price[col].apply(parse_range)

train_price[cols].head()

,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）
0,1.48,2.61,30.00
1,0.65,2.61,NaN
2,2.48,2.61,30.00
4,5.15,2.62,37.50
5,7.00,2.61,NaN


In [65]:
cols_to_fix = ['绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）', '停车费用（元）']

for col in cols_to_fix:
    train_price[col] = pd.to_numeric(train_price[col], errors='coerce')
    
# 填补缺失值
for col in cols_to_fix:
    median_value = train_price[col].median()
    train_price[col] = train_price[col].fillna(median_value)
    print(f"{col} 缺失值已用中位数 {median_value:.2f} 填补完成")

train_price[cols_to_fix].describe()


绿化率（%） 缺失值已用中位数 33.00 填补完成
容积率（倍） 缺失值已用中位数 2.50 填补完成
物业费（元/月/㎡） 缺失值已用中位数 1.88 填补完成
燃气费（元/m³） 缺失值已用中位数 2.61 填补完成
供热费（元/㎡） 缺失值已用中位数 25.00 填补完成
停车位（个） 缺失值已用中位数 756.00 填补完成
停车费用（元） 缺失值已用中位数 300.00 填补完成


,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）,停车费用（元）
count,"91,520.00","91,520.00","91,520.00","91,520.00","91,520.00","91,520.00","91,520.00"
mean,38.45,2.74,2.38,2.66,23.12,"1,025.81",322.32
std,239.78,1.37,3.45,0.52,7.86,"1,073.88",188.72
min,0.01,0.02,0.02,0.40,0.01,1.00,1.00
25%,30.00,2.10,1.48,2.46,25.00,500.00,300.00
50%,33.00,2.50,1.88,2.61,25.00,756.00,300.00
75%,35.00,3.00,2.41,2.95,25.00,"1,194.00",350.00
max,"10,500.00",30.00,76.45,5.00,50.00,"8,700.00","2,300.00"


In [66]:
# 替换异常值（IQR 方法）
for col in cols_to_fix:
    Q1 = train_price[col].quantile(0.25)
    Q3 = train_price[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    median_value = train_price[col].median()
 
    train_price[col] = train_price[col].mask((train_price[col] < lower_bound) | (train_price[col] > upper_bound), median_value)

train_price[cols_to_fix].describe()

,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）,停车费用（元）
count,"91,520.00","91,520.00","91,520.00","91,520.00","91,520.00","91,520.00","91,520.00"
mean,32.95,2.45,1.86,2.63,25.00,786.44,310.47
std,2.91,0.64,0.68,0.46,0.00,476.43,31.54
min,23.00,0.79,0.20,1.73,25.00,1.00,230.00
25%,30.20,2.13,1.48,2.46,25.00,500.00,300.00
50%,33.00,2.50,1.88,2.61,25.00,756.00,300.00
75%,35.00,2.50,2.08,2.62,25.00,885.00,300.00
max,42.30,4.33,3.81,3.67,25.00,"2,200.00",420.00


### train_rent

In [67]:
train_rent = housing['train_rent'].copy()
train_rent.drop(columns=['朝向','交易时间','车位','用水','用电','采暖','租期','配套设施','年份','物业类别','建筑年代','开发商','房屋总数','楼栋总数','物业公司','建筑结构','物业办公电话','产权描述','供水','供暖','供电','coord_x','coord_y','客户反馈','停车费用'], inplace=True)
train_rent.shape


(98899, 21)

In [68]:
train_rent.isnull().sum()

城市           0
户型           1
装修       73489
Price        0
楼层           5
面积           0
付款方式     18423
租赁方式         0
电梯           4
燃气        4582
lon          0
lat          0
区县        4677
板块        5144
环线位置     69663
绿 化 率    24402
容 积 率    24080
物 业 费    22159
燃气费      25057
供热费      70064
停车位      25479
dtype: int64

In [69]:
train_rent.dropna(subset=['电梯','区县','板块'],inplace=True)
train_rent['环线位置'] = train_rent['环线位置'].fillna('未知')
train_rent['装修'] = train_rent['装修'].fillna('非精装修')
train_rent['付款方式'] = train_rent['付款方式'].fillna('未知')
train_rent['燃气'] = train_rent['燃气'].fillna('未知')
train_rent.shape

(93751, 21)

In [70]:
train_rent.rename(columns={
    '面积': '面积（㎡）',
    '绿 化 率': '绿化率（%）',
    '容 积 率': '容积率（倍）',
    '物 业 费': '物业费（元/月/㎡）',
    '燃气费': '燃气费（元/m³）',
    '供热费': '供热费（元/㎡）',
    '停车位': '停车位（个）'
}, inplace = True)

train_rent['绿化率（%）'] = train_rent['绿化率（%）'].astype(str).str.replace('%', '', regex=False)
train_rent['绿化率（%）'] = pd.to_numeric(train_price['绿化率（%）'], errors='coerce')
train_rent['面积（㎡）'] = train_rent['面积（㎡）'].astype(str).str.replace('㎡', '', regex=False)
train_rent['燃气费（元/m³）'] = train_rent['燃气费（元/m³）'].astype(str).str.replace('元/m³', '', regex=False)
train_rent['供热费（元/㎡）'] = train_rent['供热费（元/㎡）'].astype(str).str.replace('元/㎡', '', regex=False)
train_rent['物业费（元/月/㎡）'] = train_rent['物业费（元/月/㎡）'].astype(str).str.replace(r'[^\d\.-]', '', regex=True)

train_rent.sample(5)


,城市,户型,装修,Price,楼层,面积（㎡）,付款方式,租赁方式,电梯,燃气,...,lat,区县,板块,环线位置,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）
46318,3,1室1厅1卫,精装修,"212,762.38",低楼层/20层,43.00,季付价,整租,有,有,...,32.40,64.00,484.00,未知,32.00,NaN,,nan,nan,NaN
73892,7,1室1厅1卫,非精装修,"582,921.35",高楼层/18层,28.00,月付价,整租,有,有,...,23.53,99.00,818.00,未知,33.00,5.10,3.1,3.5,nan,108.00
32730,2,2室1厅1卫,精装修,"184,299.19",中楼层/8层,88.58,季付价,整租,有,有,...,30.59,86.00,915.00,未知,34.80,4.00,0.6-1.3,1.98,nan,240.00
55963,4,3室1厅,非精装修,"863,803.49",高楼层/6层,87.61,季付价,整租,无,无,...,32.35,43.00,719.00,中环至外环,NaN,2.00,0.8,0.03-3,nan,"1,000.00"
18586,1,2室1厅,非精装修,"164,007.07",中楼层/11层,69.00,未知,整租,无,有,...,40.85,124.00,"1,113.00",未知,37.00,1.45,1.65-3.3,2.46,nan,"1,191.00"


In [71]:
# 将区间数据用平均值代替
cols = ['物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）']
for col in cols:
    train_rent[col] = train_rent[col].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
    train_rent[col] = train_rent[col].replace('', np.nan)
  
    def parse_range(x):
        if pd.isna(x):
            return np.nan
        parts = x.split('-')
        return (float(parts[0]) + float(parts[1])) / 2 if len(parts) > 1 else float(parts[0])
    
    train_rent[col] = train_rent[col].apply(parse_range)

train_rent[cols].head()

,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）
0,1.48,2.61,27.00
1,0.88,2.61,30.00
2,2.33,2.62,38.00
3,3.20,2.61,37.00
4,1.00,2.61,30.00


In [72]:
cols_to_fix1 = ['绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）']

for col in cols_to_fix1:
    train_rent[col] = pd.to_numeric(train_rent[col], errors='coerce')
    
# 填补缺失值
for col in cols_to_fix1:
    median_value = train_rent[col].median()
    train_rent[col] = train_rent[col].fillna(median_value)
    print(f"{col} 缺失值已用中位数 {median_value:.2f} 填补完成")

train_rent[cols_to_fix1].describe()


绿化率（%） 缺失值已用中位数 33.00 填补完成
容积率（倍） 缺失值已用中位数 2.70 填补完成
物业费（元/月/㎡） 缺失值已用中位数 2.20 填补完成
燃气费（元/m³） 缺失值已用中位数 2.95 填补完成
供热费（元/㎡） 缺失值已用中位数 25.00 填补完成
停车位（个） 缺失值已用中位数 760.00 填补完成


,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）
count,"93,751.00","93,751.00","93,751.00","93,751.00","93,751.00","93,751.00"
mean,32.94,2.99,2.72,2.90,23.34,"1,164.25"
std,2.71,1.57,3.51,0.52,7.70,"1,409.71"
min,23.00,0.02,0.02,0.40,0.01,1.00
25%,33.00,2.03,1.50,2.61,25.00,420.00
50%,33.00,2.70,2.20,2.95,25.00,760.00
75%,33.00,3.20,2.90,3.44,25.00,"1,300.00"
max,42.30,30.00,76.45,5.00,50.00,"8,700.00"


In [73]:
# 替换异常值（IQR 方法）
cols_to_fix2 = ['容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）',  '停车位（个）']
for col in cols_to_fix2:
    Q1 = train_rent[col].quantile(0.25)
    Q3 = train_rent[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    median_value = train_rent[col].median()
 
    train_rent[col] = train_rent[col].mask((train_rent[col] < lower_bound) | (train_rent[col] > upper_bound), median_value)

train_rent[cols_to_fix1].describe()

,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）
count,"93,751.00","93,751.00","93,751.00","93,751.00","93,751.00","93,751.00"
mean,32.94,2.62,2.16,2.89,23.34,796.44
std,2.71,0.82,0.93,0.50,7.70,533.51
min,23.00,0.30,0.02,1.50,0.01,1.00
25%,33.00,2.05,1.50,2.61,25.00,420.00
50%,33.00,2.70,2.20,2.95,25.00,760.00
75%,33.00,3.00,2.68,3.36,25.00,"1,000.00"
max,42.30,4.95,5.00,4.50,50.00,"2,604.00"


In [74]:
train_rent.drop_duplicates(inplace=True)
train_rent.shape

(93751, 21)

##  Handling Text and Categorical Attributes

### train_price

In [75]:
# 环线编码
ring_map = {
    '内环内': 1,
    '内环至中环': 2,
    '中环至外环': 3,
    '内环至外环': 3,  # 内环至外环和中环至外环视为同一层级
    '二环内': 1,
    '二至三环': 2,
    '三至四环': 3,
    '四至五环': 4,
    '五至六环': 5,
    '六环外': 6,
    '外环外': 6,
    '未知': 0
}

train_price['环线编码'] = train_price['环线'].map(ring_map)
train_price['环线编码'].fillna(0, inplace=True)
train_price[['环线', '环线编码']].head(10)


/tmp/ipykernel_37/589849513.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_price['环线编码'].fillna(0, inplace=True)


,环线,环线编码
0,二至三环,2
1,五至六环,5
2,五至六环,5
4,三至四环,3
5,五至六环,5
6,六环外,6
8,三至四环,3
9,四至五环,4
10,五至六环,5
11,四至五环,4


In [76]:
# 建筑结构编码
train_price['建筑结构'].fillna('未知结构', inplace=True)
structure_map = {
    '未知结构': 0,
    '钢混结构': 1,
    '钢结构': 2,
    '混合结构': 3,
    '框架结构': 4,  
    '砖混结构': 5,
    '砖木结构': 6,
}
train_price['建筑结构编码'] = train_price['建筑结构'].map(structure_map)
train_price[['建筑结构', '建筑结构编码']].head(10)

/tmp/ipykernel_37/3499818421.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_price['建筑结构'].fillna('未知结构', inplace=True)


,建筑结构,建筑结构编码
0,混合结构,3
1,混合结构,3
2,钢混结构,1
4,钢混结构,1
5,钢混结构,1
6,钢混结构,1
8,钢混结构,1
9,钢混结构,1
10,钢混结构,1
11,混合结构,3


In [77]:
# 房屋年限编码
# 使用众数填补缺失值
most_frequent_value = train_price['房屋年限'].mode()[0]
train_price['房屋年限'].fillna(most_frequent_value, inplace=True)
# 使用 pandas 进行独热编码
train_price = pd.get_dummies(train_price, columns=['房屋年限'], drop_first=True)

/tmp/ipykernel_37/882172506.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_price['房屋年限'].fillna(most_frequent_value, inplace=True)


In [78]:
# 楼层处理
train_price['所在楼层'] = train_price['所在楼层'].str.replace(r'\(.*\)', '', regex=True)
train_price['所在楼层'] = train_price['所在楼层'].str.strip()
floor_map = {
    '地下室': 0,
    '底层': 1,
    '低楼层': 2,
    '中楼层': 3,
    '高楼层': 4,
    '顶层': 5
}

train_price['楼层编码'] = train_price['所在楼层'].map(floor_map)
train_price['楼层编码'].head(10)

0     3
1     5
2     2
4     3
5     0
6     4
8     4
9     1
10    5
11    5
Name: 楼层编码, dtype: int64

In [79]:
import pandas as pd
import re

# 创建一个处理房屋户型的函数
def parse_layout_v2(layout):
    layout = str(layout)  # 确保传入的是字符串类型
    rooms = {'室': 0, '厅': 0, '厨': 0, '卫': 0, '房间': 0}  # 初始化各个房间数量
    match = re.findall(r'(\d+)(室|厅|厨|卫|房间)', layout) # 匹配不同类型的房间

    for num, type_ in match:
        rooms[type_] = int(num)  # 将房间数赋给对应类型

    return rooms['房间'], rooms['室'], rooms['厅'], rooms['厨'], rooms['卫']

# 确保 '房屋户型' 列的数据为字符串格式，并填充缺失值
train_price['房屋户型'] = train_price['房屋户型'].fillna('未知').astype(str)

# 应用到数据
train_price[['房间数', '室数', '厅数', '厨数', '卫数']] = train_price['房屋户型'].apply(lambda x: pd.Series(parse_layout_v2(x)))

# 检查处理后的数据
print(train_price[['房屋户型', '房间数', '室数', '厅数', '厨数', '卫数']].head())


       房屋户型  房间数  室数  厅数  厨数  卫数
0  2室1厅1厨1卫    0   2   1   1   1
1  3室1厅1厨1卫    0   3   1   1   1
2  3室2厅1厨2卫    0   3   2   1   2
4     1房间1卫    1   0   0   0   1
5  5室2厅1厨4卫    0   5   2   1   4


In [80]:
# 配备电梯处理
train_price['有电梯'] = train_price['配备电梯'].map({'无': 0, '有': 1})
train_price['有电梯'].head()

0    0
1    0
2    1
4    1
5    0
Name: 有电梯, dtype: int64

In [81]:
train_price.drop(columns=['环线','所在楼层','配备电梯','建筑结构','房屋户型'], inplace=True)

### train_rent

In [82]:
# 装修
train_rent['精装修'] = train_rent['装修'].map({'非精装修': 0, '精装修': 1})

# 付款方式
print(train_rent['付款方式'].unique())
train_rent['付款方式'] = train_rent['付款方式'].str.replace(r'http\S+', '未知', regex=True)
print(train_rent['付款方式'].unique())
train_rent = pd.get_dummies(train_rent, columns=['付款方式'], drop_first=True)
train_rent.columns

['季付价' '未知' '年付价' '半年付价' '月付价' '双月付价' 'https://image1.ljcdn.com/rent-'
 'https://img.ljcdn.com/usercent']
['季付价' '未知' '年付价' '半年付价' '月付价' '双月付价']


Index(['城市', '户型', '装修', 'Price', '楼层', '面积（㎡）', '租赁方式', '电梯', '燃气', 'lon',
       'lat', '区县', '板块', '环线位置', '绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）',
       '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）', '精装修', '付款方式_双月付价', '付款方式_季付价',
       '付款方式_年付价', '付款方式_月付价', '付款方式_未知'],
      dtype='object')

In [83]:
# 租赁方式
train_rent['整租'] = train_rent['租赁方式'].map({'合租': 0, '整租': 1})
# 电梯
train_rent['有电梯'] = train_rent['电梯'].map({'无': 0, '有': 1})
# 燃气
train_rent['有燃气'] = train_rent['燃气'].map({'无': 0,'未知': 0, '有': 1})
# 环线编码
ring_map = {
    '内环内': 1,
    '内环至中环': 2,
    '中环至外环': 3,
    '内环至外环': 3,  
    '二环内': 1,
    '二至三环': 2,
    '三至四环': 3,
    '四至五环': 4,
    '五至六环': 5,
    '六环外': 6,
    '外环外': 6,
    '未知': 0
}
train_rent['环线编码'] = train_rent['环线位置'].map(ring_map)
train_rent['环线编码'].fillna(0, inplace=True)

/tmp/ipykernel_37/3142996470.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_rent['环线编码'].fillna(0, inplace=True)


In [84]:
# 户型
# 确保 '房屋户型' 列的数据为字符串格式，并填充缺失值
train_rent['户型'] = train_rent['户型'].fillna('未知').astype(str)

# 应用到数据
train_rent[['房间数', '室数', '厅数', '厨数', '卫数']] = train_rent['户型'].apply(lambda x: pd.Series(parse_layout_v2(x)))

# 检查处理后的数据
print(train_rent[['户型', '房间数', '室数', '厅数', '厨数', '卫数']].head())


       户型  房间数  室数  厅数  厨数  卫数
0  1室1厅1卫    0   1   1   0   1
1  1室1厅1卫    0   1   1   0   1
2  1室1厅1卫    0   1   1   0   1
3  3室1厅2卫    0   3   1   0   2
4  1室1厅1卫    0   1   1   0   1


In [85]:
# 楼层
# 定义楼层区间
def convert_floor(floor_data):
    floor_data = str(floor_data).strip()
    # 如果是地下室
    if '地下' in str(floor_data):
        return '地下室'
    # 处理 'x/y层' 格式的楼层数据
    match = re.match(r'(\d+)/(\d+)层', str(floor_data))
    if match:
        floor_num = int(match.group(1))
        total_floors = int(match.group(2))
        # 判断低楼层、中楼层和高楼层
        if floor_num <= total_floors * 0.3:  # 低楼层
            return '低楼层'
        elif floor_num <= total_floors * 0.7:  # 中楼层
            return '中楼层'
        else:  # 高楼层
            return '高楼层'
    # 3. 处理 '低楼层/x楼'、'中楼层/x楼'、'高楼层/x楼' 格式的数据
    match = re.match(r'(低楼层|中楼层|高楼层)/(\d+)楼', floor_data)
    if match:
        return match.group(1)  # 返回对应的楼层类型
    
    
    # 对于其他情况，返回 '未知'
    return '未知'

# 应用转换函数
train_rent['楼层类型'] = train_rent['楼层'].apply(convert_floor)

# 将楼层类型转化为编码
floor_mapping = {'低楼层': 1, '中楼层': 2, '高楼层': 3, '地下室': 4, '未知': 5}
train_rent['楼层类型编码'] = train_rent['楼层类型'].map(floor_mapping)

# 查看结果
train_rent['楼层类型编码'].head()

0    2
1    2
2    1
3    1
4    3
Name: 楼层类型编码, dtype: int64

In [86]:
train_rent.drop(columns=['装修','租赁方式','电梯','燃气','环线位置','户型', '楼层类型','楼层'], inplace=True)
train_rent['城市'] = pd.to_numeric(train_rent['城市'], errors='coerce')
train_rent['面积（㎡）'] = pd.to_numeric(train_rent['面积（㎡）'], errors='coerce')
train_rent[['付款方式_双月付价','付款方式_季付价','付款方式_年付价','付款方式_月付价','付款方式_未知']] = train_rent[['付款方式_双月付价','付款方式_季付价','付款方式_年付价','付款方式_月付价','付款方式_未知']].astype(int)

train_rent.dtypes

城市              int64
Price         float64
面积（㎡）         float64
lon           float64
lat           float64
区县            float64
板块            float64
绿化率（%）        float64
容积率（倍）        float64
物业费（元/月/㎡）    float64
燃气费（元/m³）     float64
供热费（元/㎡）      float64
停车位（个）        float64
精装修             int64
付款方式_双月付价       int64
付款方式_季付价        int64
付款方式_年付价        int64
付款方式_月付价        int64
付款方式_未知         int64
整租              int64
有电梯             int64
有燃气             int64
环线编码            int64
房间数             int64
室数              int64
厅数              int64
厨数              int64
卫数              int64
楼层类型编码          int64
dtype: object

# Feature Engineering

## Skewness

In [87]:
# 计算偏度
skewed_features = train_price.select_dtypes(include=[np.number]).skew()

# 查看偏度较大的特征（绝对值大于1）
skewed_features = skewed_features[skewed_features.abs() > 1]
print(skewed_features)

Price      4.09
建筑面积（㎡）    1.85
停车费用（元）    1.84
环线编码       1.42
建筑结构编码     2.39
房间数       15.41
厨数        -2.00
卫数         1.59
有电梯       -1.31
dtype: float64


In [88]:
cols = ['Price', '建筑面积（㎡）','环线编码', '建筑结构编码']
train_price['log_price'] = np.log1p(train_price['Price'])
train_price['log_area'] = np.log1p(train_price['建筑面积（㎡）'])
train_price['log_ringcode'] = np.log1p(train_price['环线编码'])
train_price['log_stru_code'] = np.log1p(train_price['建筑结构编码'])
for col in ['log_price', 'log_area', 'log_ringcode','log_stru_code']:
    print(col, '偏度 =', train_price[col].skew())



log_price 偏度 = 0.28872458021975933
log_area 偏度 = -0.1268566205136206
log_ringcode 偏度 = 0.853491320319867
log_stru_code 偏度 = 1.6417965577491822


In [89]:
cols = ['Price', '面积（㎡）','卫数']
for col in cols:
    print(col, '偏度 =', train_rent[col].skew())

Price 偏度 = 5.215102536424168
面积（㎡） 偏度 = 1.633344967567743
卫数 偏度 = 1.6031481509854535


In [90]:
train_rent['log_price'] = np.log1p(train_rent['Price'])
train_rent['log_area'] = np.log1p(train_rent['面积（㎡）'])
train_rent['log_bathroom'] = np.log1p(train_rent['卫数'])

for col in ['log_price', 'log_area', 'log_bathroom']:
    print(col, '偏度 =', train_rent[col].skew())

log_price 偏度 = 0.3709961800417537
log_area 偏度 = -0.9612559334777454
log_bathroom 偏度 = 0.9619349746222539


## interaction

In [91]:
train_price.columns

Index(['城市', '区域', '板块', 'Price', '建筑面积（㎡）', 'lon', 'lat', '绿化率（%）', '容积率（倍）',
       '物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）', '停车费用（元）', '环线编码',
       '建筑结构编码', '房屋年限_满两年', '房屋年限_满五年', '楼层编码', '房间数', '室数', '厅数', '厨数', '卫数',
       '有电梯', 'log_price', 'log_area', 'log_ringcode', 'log_stru_code'],
      dtype='object')

In [92]:
train_rent.columns

Index(['城市', 'Price', '面积（㎡）', 'lon', 'lat', '区县', '板块', '绿化率（%）', '容积率（倍）',
       '物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）', '精装修', '付款方式_双月付价',
       '付款方式_季付价', '付款方式_年付价', '付款方式_月付价', '付款方式_未知', '整租', '有电梯', '有燃气',
       '环线编码', '房间数', '室数', '厅数', '厨数', '卫数', '楼层类型编码', 'log_price',
       'log_area', 'log_bathroom'],
      dtype='object')

In [93]:
train_price['建筑面积_绿化率'] = train_price['建筑面积（㎡）'] * train_price['绿化率（%）']
train_price['城市_建筑面积'] = train_price['城市'] * train_price['建筑面积（㎡）']
train_price['板块_容积率'] = train_price['板块'] * train_price['容积率（倍）']
train_price['log_area_房间数'] = train_price['log_area'] * train_price['房间数'] 


In [94]:
train_rent['城市_面积'] = train_rent['城市'] * train_rent['面积（㎡）']
train_rent['整租_房间数'] = train_rent['整租'] * train_rent['房间数']
train_rent['有电梯_楼层类型'] = train_rent['有电梯'] * train_rent['楼层类型编码']

## Binning

In [95]:
df1 = train_price.copy()

# 自定义分箱区间
bins = [0, 300, 400, df1['停车费用（元）'].max()]  # 分成三个区间：低、中、高
labels = ['低停车费', '中停车费', '高停车费']  # 对应标签
# 创建分箱列
df1['停车费用_bin'] = pd.cut(df1['停车费用（元）'], bins=bins, labels=labels, include_lowest=True)
# 查看结果
print(df1[['停车费用（元）', '停车费用_bin']].sample(5))
# One-Hot 编码
df1_dummies = pd.get_dummies(df1['停车费用_bin'], prefix='停车费用', drop_first=True)
df1 = pd.concat([df1, df1_dummies], axis=1)
# 查看编码后的结果
print(df1.sample(5))

       停车费用（元） 停车费用_bin
72275   300.00     低停车费
21015   300.00     低停车费
57144   300.00     低停车费
16065   300.00     低停车费
21158   300.00     低停车费
       城市     区域       板块         Price  建筑面积（㎡）    lon   lat  绿化率（%）  容积率（倍）  \
9222    0   7.00   931.00 10,949,619.98   274.87 117.40 40.86   30.00    2.33   
96966  10  32.00 1,182.00  5,075,548.98   125.22 114.40 24.20   33.00    2.10   
23847   2  78.00   349.00    617,305.13    74.00 107.32 30.60   40.00    2.50   
81112   8 110.00   883.00  2,116,047.18   183.00 103.70 26.02   33.00    1.46   
52014   3  20.00   905.00    552,263.67    57.60 121.55 32.25   33.00    2.50   

       物业费（元/月/㎡）  ...  log_area  log_ringcode  log_stru_code  建筑面积_绿化率  \
9222         2.17  ...      5.62          1.39           0.69  8,246.10   
96966        2.10  ...      4.84          0.00           0.69  4,132.26   
23847        1.88  ...      4.32          1.39           0.69  2,960.00   
81112        1.75  ...      5.21          0.00           1.61  6,039.

In [96]:
df2 = train_rent.copy()
# 自定义分箱
bins = [0, 20, 30, df2['供热费（元/㎡）'].max()]
labels = ['低供热费', '中供热费', '高供热费']

df2['供热费_bin'] = pd.cut(df2['供热费（元/㎡）'], bins=bins, labels=labels, include_lowest=True)

# One-Hot 编码
df2_dummies = pd.get_dummies(df2['供热费_bin'], prefix='供热费', drop_first=True)
df2 = pd.concat([df2, df2_dummies], axis=1)

# 查看结果
print(df2[['供热费（元/㎡）', '供热费_bin', '供热费_中供热费', '供热费_高供热费']].head())


   供热费（元/㎡） 供热费_bin  供热费_中供热费  供热费_高供热费
0     27.00    中供热费      True     False
1     30.00    中供热费      True     False
2     38.00    高供热费     False      True
3     37.00    高供热费     False      True
4     30.00    中供热费      True     False


# Feature Selection

In [97]:
df1.drop(columns=['Price', '建筑面积（㎡）','停车费用（元）','停车费用_bin','供热费（元/㎡）'],inplace=True)
df1['停车费用_中停车费'] = df1['停车费用_中停车费'].astype(int)  # 转换为整数 0 或 1
df1['停车费用_高停车费'] = df1['停车费用_高停车费'].astype(int) 

In [98]:
df2.drop(columns=['Price', '面积（㎡）','供热费（元/㎡）','供热费_bin'],inplace=True)
df2['供热费_中供热费'] = df2['供热费_中供热费'].astype(int)  # 转换为整数 0 或 1
df2['供热费_高供热费'] = df2['供热费_高供热费'].astype(int) 

In [99]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# 特征和目标
X1 = train_price.drop(columns=['log_price', 'Price'])  
y1 = train_price['log_price']

# 标准化数据
scaler = StandardScaler()
X1_scaled = scaler.fit_transform(X1)

# 切分数据集
X1_train, X1_test, y1_train, y1_test = train_test_split(X1_scaled, y1, test_size=0.2, random_state=111)

# LassoCV 自动选择 alpha
lasso_cv = LassoCV(alphas=[0.01,0.1, 1, 10, 100], cv=5)  # 使用5折交叉验证
lasso_cv.fit(X1_train, y1_train)

# 打印最优的 alpha 和系数
print("最佳 alpha:", lasso_cv.alpha_)
print("LASSO 回归系数：", lasso_cv.coef_)

# 获取系数不为零的特征
selected_features = X1.columns[lasso_cv.coef_ != 0]
print("Selected features by LASSO:", selected_features)

# 在测试集上进行预测
y1_pred = lasso_cv.predict(X1_test)

# 计算 MAE（Mean Absolute Error）
mae = mean_absolute_error(y1_test, y1_pred)
print("Mean Absolute Error (MAE):", mae)

最佳 alpha: 0.01
LASSO 回归系数： [-0.16540783  0.02900561 -0.00985527  0.08526611  0.12751418 -0.
 -0.0333116   0.04081477  0.05198753  0.34802614  0.         -0.0209647
  0.0008229  -0.18424826 -0.          0.          0.         -0.00418082
 -0.01525266  0.         -0.04084187  0.01350861  0.02867476  0.00588593
  0.24502858  0.40792534  0.01839068  0.04705378 -0.         -0.
 -0.        ]
Selected features by LASSO: Index(['城市', '区域', '板块', '建筑面积（㎡）', 'lon', '绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）',
       '燃气费（元/m³）', '停车位（个）', '停车费用（元）', '环线编码', '楼层编码', '房间数', '厅数', '厨数',
       '卫数', '有电梯', 'log_area', 'log_ringcode', 'log_stru_code', '建筑面积_绿化率'],
      dtype='object')
Mean Absolute Error (MAE): 0.4112117283906747


In [100]:
# 特征和目标
X2 = train_rent.drop(columns=['log_price','Price'])  
y2 = train_rent['log_price']

# 标准化数据
scaler = StandardScaler()
X2_scaled = scaler.fit_transform(X2)

# 切分数据集
X2_train, X2_test, y2_train, y2_test = train_test_split(X2_scaled, y2, test_size=0.2, random_state=111)

# LassoCV 自动选择 alpha
lasso_cv = LassoCV(alphas=[0.01,0.1, 1, 10, 100], cv=5)  # 使用5折交叉验证
lasso_cv.fit(X2_train, y2_train)

# 打印最优的 alpha 和系数
print("最佳 alpha:", lasso_cv.alpha_)
print("LASSO 回归系数：", lasso_cv.coef_)

# 获取系数不为零的特征
selected_features = X2.columns[lasso_cv.coef_ != 0]
print("Selected features by LASSO:", selected_features)

# 在测试集上进行预测
y2_pred = lasso_cv.predict(X2_test)

# 计算 MAE（Mean Absolute Error）
mae = mean_absolute_error(y2_test, y2_pred)
print("Mean Absolute Error (MAE):", mae)

最佳 alpha: 0.01
LASSO 回归系数： [-0.08144737  0.29565815  0.05388034 -0.          0.03738249 -0.04386516
  0.          0.0134159   0.05653153  0.19336401  0.03847161 -0.08243015
  0.02223252  0.          0.07835211 -0.00746331  0.09777813 -0.07489684
  0.11057084  0.          0.03086567  0.22786557  0.         -0.
 -0.01436285  0.         -0.         -0.05059901  0.         -0.
 -0.          0.          0.04491503]
Selected features by LASSO: Index(['城市', '面积（㎡）', 'lon', '区县', '板块', '容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）',
       '供热费（元/㎡）', '停车位（个）', '精装修', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价',
       '付款方式_未知', '整租', '有燃气', '环线编码', '厅数', '楼层类型编码', '有电梯_楼层类型'],
      dtype='object')
Mean Absolute Error (MAE): 0.39467222787183237


# Modeling

In [101]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd
import joblib  # 用于保存和加载模型

# 设置 pandas 显示选项，避免科学计数法
pd.set_option('display.float_format', '{:,.2f}'.format)

# 1. 准备数据
X1 = train_price[['城市', '区域', '板块', '建筑面积（㎡）', 'lon', '绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）',
       '燃气费（元/m³）', '停车位（个）', '停车费用（元）', '环线编码', '楼层编码', '房间数', '厅数', '厨数',
       '卫数', '有电梯', 'log_area', 'log_ringcode', 'log_stru_code', '建筑面积_绿化率']]  # 使用Lasso筛选后的特征
y1 = train_price['log_price']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=111)

# 2. 定义模型与参数
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet()
}

param_grids = {
    "Ridge": {"alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso": {"alpha": [0.001, 0.01, 0.1, 1, 10]},
    "ElasticNet": {"alpha": [0.01, 0.1, 1, 10], "l1_ratio": [0.2, 0.5, 0.8]}
}

# 3. 建模与评估
results = []
best_model_OLS = None
best_model_Lasso = None
best_model_Ridge = None
best_model_ElasticNet = None  # 定义ElasticNet模型变量

# 训练和评估每个模型
for name, model in models.items():
    if name in param_grids:
        grid = GridSearchCV(model, param_grids[name], cv=6, scoring='neg_mean_absolute_error')
        grid.fit(X1_train, y1_train)
        best_model = grid.best_estimator_
    else:
        best_model = model.fit(X1_train, y1_train)
    
    # 保存模型（OLS, Lasso, Ridge, ElasticNet）
    if name == 'OLS':
        best_model_OLS = best_model
    elif name == 'Lasso':
        best_model_Lasso = best_model
    elif name == 'Ridge':
        best_model_Ridge = best_model
    elif name == 'ElasticNet':
        best_model_ElasticNet = best_model  # 保存ElasticNet模型
    
    # 预测
    y1_train_pred = best_model.predict(X1_train)
    y1_test_pred = best_model.predict(X1_test)
    
    # MAE（对数值）
    mae_train = mean_absolute_error(y1_train, y1_train_pred)
    mae_test = mean_absolute_error(y1_test, y1_test_pred)
    mae_cv = -cross_validate(best_model, X1_train, y1_train, cv=6, scoring='neg_mean_absolute_error')['test_score'].mean()
    
    # 将对数价格的 MAE 添加到结果中
    results.append({
        "Model": name,
        "Train MAE (Log)": mae_train,
        "Test MAE (Log)": mae_test,
        "CV MAE (Log)": mae_cv
    })

# 4. 将对数变换值反转回原始值，并计算原始房价的 MAE
# 反转 y2_train 和 y2_test 的对数值
y1_train_original = np.exp(y1_train)
y1_test_original = np.exp(y1_test)

# 对每个模型进行预测后，将预测的对数值反转回原始值
y1_train_pred_original = np.exp(y1_train_pred)
y1_test_pred_original = np.exp(y1_test_pred)

# 计算原始值上的 MAE
mae_train_original = mean_absolute_error(y1_train_original, y1_train_pred_original)
mae_test_original = mean_absolute_error(y1_test_original, y1_test_pred_original)

# 5. 计算原始值的 CV MAE
y1_train_original_cv = np.exp(y1_train)
mae_cv_original = -cross_validate(best_model, X1_train, y1_train_original_cv, cv=6, scoring='neg_mean_absolute_error')['test_score'].mean()

# 将原始 MAE 添加到结果中
for result in results:
    result["Train MAE (Original)"] = mae_train_original
    result["Test MAE (Original)"] = mae_test_original
    result["CV MAE (Original)"] = mae_cv_original

# 6. 打印最终的结果表
results_df = pd.DataFrame(results)
print(results_df)

# 7. 保存训练好的模型到文件（使用joblib）
joblib.dump(best_model_OLS, 'model_OLS.pkl')  # 保存OLS模型
joblib.dump(best_model_Lasso, 'model_Lasso.pkl')  # 保存Lasso模型
joblib.dump(best_model_Ridge, 'model_Ridge.pkl')  # 保存Ridge模型
joblib.dump(best_model_ElasticNet, 'model_ElasticNet.pkl')  # 保存ElasticNet模型


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.002e+01, tolerance: 4.078e+00
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.437e+01, tolerance: 4.062e+00
  model = cd_fast.enet_coordinate_descent(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.951e+01, tolerance: 4.055e

        Model  Train MAE (Log)  Test MAE (Log)  CV MAE (Log)  \
0         OLS             0.41            0.41          0.41   
1       Ridge             0.41            0.41          0.41   
2       Lasso             0.41            0.41          0.41   
3  ElasticNet             0.41            0.41          0.41   

   Train MAE (Original)  Test MAE (Original)  CV MAE (Original)  
0            959,832.53           938,794.52       1,145,991.94  
1            959,832.53           938,794.52       1,145,991.94  
2            959,832.53           938,794.52       1,145,991.94  
3            959,832.53           938,794.52       1,145,991.94  


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_coordinate_descent.py:631: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.018e+17, tolerance: 3.858e+13
  model = cd_fast.enet_coordinate_descent(


['model_ElasticNet.pkl']

In [104]:
# 设置 pandas 显示选项，避免科学计数法
pd.set_option('display.float_format', '{:,.2f}'.format)

# 1. 准备数据
X2 = train_rent[['城市', '面积（㎡）', 'lon', '容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）',
       '供热费（元/㎡）', '停车位（个）', '精装修', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价',
       '付款方式_未知', '整租', '有燃气', '环线编码', '厅数', '楼层类型编码', '有电梯_楼层类型']]  # 使用Lasso筛选后的特征
y2 = train_rent['log_price']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=111)

# 2. 定义模型与参数
models = {
    "OLS": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet()
}

param_grids = {
    "Ridge": {"alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso": {"alpha": [0.001, 0.01, 0.1, 1, 10]},
    "ElasticNet": {"alpha": [0.01, 0.1, 1, 10], "l1_ratio": [0.2, 0.5, 0.8]}
}

# 3. 建模与评估
results = []
saved_models = {}

for name, model in models.items():
    if name in param_grids:
        grid = GridSearchCV(model, param_grids[name], cv=6, scoring='neg_mean_absolute_error')
        grid.fit(X2_train, y2_train)
        fitted_model = grid.best_estimator_
    else:
        fitted_model = model.fit(X2_train, y2_train)
    
    # 保存模型
    saved_models[name] = fitted_model
    
    # 预测
    y_train_pred = fitted_model.predict(X2_train)
    y_test_pred = fitted_model.predict(X2_test)
    
    # 计算对数 MAE
    mae_train_log = mean_absolute_error(y2_train, y_train_pred)
    mae_test_log = mean_absolute_error(y2_test, y_test_pred)
    
    # 交叉验证 MAE（对数值）
    mae_cv_log = -cross_validate(fitted_model, X2_train, y2_train, cv=6, scoring='neg_mean_absolute_error')['test_score'].mean()
    
    # 原始值预测
    y_train_pred_original = np.exp(y_train_pred)
    y_test_pred_original = np.exp(y_test_pred)
    y_train_original = np.exp(y2_train)
    y_test_original = np.exp(y2_test)
    
    mae_train_original = mean_absolute_error(y_train_original, y_train_pred_original)
    mae_test_original = mean_absolute_error(y_test_original, y_test_pred_original)
    
    # 交叉验证 MAE（原始值）
    def exp_predict(model, X):
        return np.exp(model.predict(X))
    
    cv_scores_original = []
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=6, shuffle=True, random_state=111)
    for train_index, val_index in kf.split(X2_train):
        X_tr, X_val = X2_train.iloc[train_index], X2_train.iloc[val_index]
        y_tr, y_val = y2_train.iloc[train_index], y2_train.iloc[val_index]
        fitted_model.fit(X_tr, y_tr)
        y_val_pred_original = np.exp(fitted_model.predict(X_val))
        y_val_original = np.exp(y_val)
        cv_scores_original.append(mean_absolute_error(y_val_original, y_val_pred_original))
    mae_cv_original = np.mean(cv_scores_original)
    
    # 添加结果
    results.append({
        "Model": name,
        "Train MAE (Log)": mae_train_log,
        "Test MAE (Log)": mae_test_log,
        "CV MAE (Log)": mae_cv_log,
        "Train MAE (Original)": mae_train_original,
        "Test MAE (Original)": mae_test_original,
        "CV MAE (Original)": mae_cv_original
    })

# 4. 打印结果
results_df = pd.DataFrame(results)
print(results_df)

# 5. 保存模型
for name, model in saved_models.items():
    joblib.dump(model, f'model_{name}_rent.pkl')


        Model  Train MAE (Log)  Test MAE (Log)  CV MAE (Log)  \
0         OLS             0.40            0.40          0.40   
1       Ridge             0.40            0.40          0.40   
2       Lasso             0.40            0.40          0.40   
3  ElasticNet             0.40            0.40          0.40   

   Train MAE (Original)  Test MAE (Original)  CV MAE (Original)  
0            251,299.34           244,237.22         251,381.44  
1            251,299.36           244,237.23         251,381.45  
2            251,553.85           244,558.08         251,633.86  
3            253,005.24           246,179.01         253,083.29  
